# Resultados

Lê `resultados/registro.csv` e monta as tabelas de ablação. Não roda modelos.
Definições de V, T e suportes em `PROTOCOLO.md`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
from protovm import registro

pd.set_option('display.precision', 3)
df = registro.tabela()
cols = ['id', 'modelo', 'pesos', 'passo', 'treino_resolucao', 'aval_estagio', 'aval_norm', 'aval_resolucao', 'grade', 'dice_alp', 'dice_sam', 'gpu', 'obs']
print(len(df), 'avaliações')

## T (teste Kvasir, suporte S00)

In [ ]:
df[(df.conjunto == 'T')][cols]

## V, suporte S00: encoders e EFT

In [ ]:
v = df[(df.conjunto == 'V') & (df.suporte == 'S00')]
v[cols].sort_values(['modelo', 'pesos', 'aval_estagio', 'passo'])

## Curva do EFT por passo (V, S00)

In [ ]:
eft = v[v.pesos.str.startswith('eft') & (v.passo > 0)]
eft.pivot_table(index='passo', columns=['pesos', 'treino_resolucao', 'aval_estagio', 'aval_norm', 'aval_resolucao'], values='dice_sam', aggfunc='last')

## Robustez ao suporte (20 suportes em V)

In [ ]:
multi = df[(df.conjunto == 'V') & df.obs.str.contains('exp. 15')]
par = multi.pivot_table(index='suporte', columns='pesos', values='dice_sam')
par.columns = ['EFT 2000' if c.startswith('eft') else 'pré-treinado' for c in par.columns]
par['ganho'] = par['EFT 2000'] - par['pré-treinado']
display(par)
par.agg(['mean', 'std'])